# CIS 5370 Final Project

## Project 1 - Intrusion Detection for Industrial Control Systems

- Kyle Brindle
- Rayhan Alcena

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import sklearn
from sklearn.ensemble import IsolationForest
from nfstream import NFStreamer

## Loading Files

In [14]:
# Base path to the datasets
BASE_PATH = Path.cwd() / "data/SWaT_A6_Dec_2019/"
SENSOR_READINGS_PATH = BASE_PATH / "csv/"
NETWORK_READINGS_PATH = BASE_PATH / "pcap/"

# Sensor Readings CSV/XLSX file paths
dec_2019_file_path = SENSOR_READINGS_PATH / "Dec2019.xlsx"

# Network Readings CSV/XLSX file paths
test_converted_pcap_file_path = NETWORK_READINGS_PATH / "pcap_00_converted.csv"

# Loading the CSV/XLSX files (We had to skip the first 9 rows (0-9))
sensor_readings_df = pd.read_excel(dec_2019_file_path, skiprows=9)
network_readings_df = pd.read_csv(test_converted_pcap_file_path)

## Preprocessing

In [ ]:
flows_count = NFStreamer(source=NETWORK_READINGS_PATH / "Dec2019_00000_20191206100500.pcap").to_csv(path=test_converted_pcap_file_path)

## Sensor Readings Evaluations

In [8]:
sensor_readings_df.head(10)

,t_stamp,P1_STATE,LIT101.Pv,FIT101.Pv,MV101.Status,P101.Status,P102.Status,P2_STATE,FIT201.Pv,AIT201.Pv,...,FIT601.Pv,P601.Status,P602.Status,P603.Status,LSH601.Alarm,LSL601.Alarm,LSH602.Alarm,LSL602.Alarm,LSH603.Alarm,LSL603.Alarm
0,2019-12-06 10:05:00,3,658.661255,0.0,1,2.0,1,2,2.313523,35.215330,...,0.000256,2,1,1,Inactive,Inactive,Active,Inactive,Inactive,Active
1,2019-12-06 10:05:01,3,659.171600,0.0,1,2.0,1,2,2.311857,35.215330,...,0.000256,2,1,1,Inactive,Inactive,Active,Inactive,Inactive,Active
2,2019-12-06 10:05:02,3,659.681800,0.0,1,2.0,1,2,2.311601,35.215330,...,0.000256,2,1,1,Inactive,Inactive,Active,Inactive,Inactive,Active
3,2019-12-06 10:05:03,3,660.349100,0.0,1,2.0,1,2,2.310448,35.215330,...,0.000256,2,1,1,Inactive,Inactive,Active,Inactive,Inactive,Active
4,2019-12-06 10:05:04,3,660.780945,0.0,1,2.0,1,2,2.310448,35.215330,...,0.000256,2,1,1,Inactive,Inactive,Active,Inactive,Inactive,Active
5,2019-12-06 10:05:05,3,661.016400,0.0,1,2.0,1,2,2.311088,35.087160,...,0.000256,2,1,1,Inactive,Inactive,Active,Inactive,Inactive,Active
6,2019-12-06 10:05:06,3,661.055700,0.0,1,2.0,1,2,2.316086,34.958984,...,0.000256,2,1,1,Inactive,Inactive,Active,Inactive,Inactive,Active
7,2019-12-06 10:05:07,3,661.330444,0.0,1,2.0,1,2,2.317624,34.958984,...,0.000256,2,1,1,Inactive,Inactive,Active,Inactive,Inactive,Active
8,2019-12-06 10:05:08,3,661.369700,0.0,1,2.0,1,2,2.321469,34.958984,...,0.000256,2,1,1,Inactive,Inactive,Active,Inactive,Inactive,Active
9,2019-12-06 10:05:09,3,661.644500,0.0,1,2.0,1,2,2.325057,34.958984,...,0.000256,2,1,1,Inactive,Inactive,Active,Inactive,Inactive,Active


## Network Readings Evaluations

In [15]:
network_readings_df.head(100000)

,No.,Time,Source,Destination,Protocol,Length,Info
0,1,0.000000,192.168.1.10,192.168.1.30,CIP,134,'HMI_PLANT_RESET' - Service (0x4d)
1,2,0.000001,192.168.1.10,192.168.1.30,CIP,124,'HMI_P3_PERMISSIVE' - Service (0x4c)
2,3,0.000002,192.168.1.10,192.168.1.30,TCP,128,"[TCP Retransmission] 55625 > 44818 [PSH, ACK..."
3,4,0.000003,192.168.1.30,192.168.1.100,CIP,280,Success: Service (0x4c)
4,5,0.000004,192.168.1.30,192.168.1.20,CIP,126,'HMI_P_NAOCL_UF_DUTY' - Service (0x4c)
...,...,...,...,...,...,...,...
99995,99996,4.067051,192.168.1.220,192.168.1.200,TCP,66,"47734 > 443 [FIN, ACK] Seq=826 Ack=968 Win=3..."
99996,99997,4.067052,192.168.1.220,192.168.1.200,TCP,70,"[TCP Retransmission] 47734 > 443 [FIN, ACK] ..."
99997,99998,4.067216,192.168.1.10,192.168.1.60,TCP,60,56295 > 44818 [ACK] Seq=44705 Ack=31999 Win=...
99998,99999,4.067217,192.168.1.10,192.168.1.60,TCP,64,[TCP Dup ACK 99998#1] 56295 > 44818 [ACK] Se...


## Isolation Forest

In [18]:
cutoff = network_readings_df["Time"].quantile(0.7)

train = network_readings_df[network_readings_df["Time"] <= cutoff]
test = network_readings_df[network_readings_df["Time"] > cutoff]

model = IsolationForest(contamination=0.01)
model.fit(train)
scores = model.decision_function(test)
preds = model.predict(test)
df_test = pd.DataFrame(test)
df_test["anomaly_scores"] = scores
df_test["prediction"] = preds
df_test.head(100)

ValueError: could not convert string to float: '192.168.1.10'